In [2]:
import pandas as pd

Exception ignored in PyObject_HasAttr(); consider using PyObject_HasAttrWithError(), PyObject_GetOptionalAttr() or PyObject_GetAttr():
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 491, in _call_with_frames_removed
AttributeError: partially initialized module 'pandas' from 'C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\pandas\__init__.py' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)


AttributeError: partially initialized module 'pandas' from 'C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\pandas\__init__.py' has no attribute 'core' (most likely due to a circular import)

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")

In [ ]:
df.head()

In [ ]:
df.drop(columns = ["RowNumber", "CustomerId", "Surname"],inplace = True)

In [ ]:
df.head()

In [ ]:
df["Geography"].value_counts()

In [ ]:
df["Gender"].value_counts()

In [ ]:
df = pd.get_dummies(df,columns=['Geography','Gender'],drop_first=True)

In [ ]:
df.head()

In [ ]:
X = df.drop(columns=['Exited'])
y = df['Exited'].values

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_trf = scaler.fit_transform(X_train)
X_test_trf = scaler.transform(X_test)

In [ ]:
df.head()

In [ ]:
X_train_trf.shape

In [ ]:
len(df.columns.to_list())

In [ ]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(11,activation="relu"),
    keras.layers.Dense(1,activation="sigmoid"),
    ]

)

In [ ]:
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
model.fit(X_train_trf,y_train,batch_size=50,epochs=100,verbose=1,validation_split=0.2)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
y_pred = model.predict(X_test_trf)
y_pred_classes = (y_pred > 0.5).astype(int).ravel()

print(confusion_matrix(y_test, y_pred_classes))
print(classification_report(y_test, y_pred_classes))

In [ ]:
print(y_pred.min(), y_pred.max(), y_pred.mean())

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

def report(name, y_true, proba, threshold=0.5):
    pred = (proba > threshold).astype(int)
    print(f"\n===== {name} =====")
    print(f"ROC-AUC : {roc_auc_score(y_true, proba):.3f}")
    print(f"PR-AUC  : {average_precision_score(y_true, proba):.3f}")
    print(classification_report(y_true, pred, digits=3))

# ---------- 1. Logistic regression ----------
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train_trf, y_train)
report("Logistic regression", y_test, logreg.predict_proba(X_test_trf)[:, 1])

# ---------- 2. LightGBM ----------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_trf, y_train, test_size=0.2, stratify=y_train, random_state=42
)

gbm = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=42,
    verbose=-1,
)
gbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)
print(f"\nLightGBM stopped at iteration {gbm.best_iteration_} of 2000")
report("LightGBM", y_test, gbm.predict_proba(X_test_trf)[:, 1])

# ---------- 3. Keras ----------
keras_proba = model.predict(X_test_trf, verbose=0).ravel()
report("Keras", y_test, keras_proba)